# 리포트 02 — 경로는 SBR 과 이미지법이 정한다

> ### 한 일
> **Sionna RT 설치본의 경로 탐색을 소스 줄 번호까지 읽어 광선이 무엇을 하고 어디서 경로가 확정되는지를 적었다.**

### 결과
1. 소스마다 광선을 피보나치 격자로 뿌려 면 순서 후보를 모으고, 같은 면 순서를 발견한 광선은 첫 발만 남긴다(`sb_candidate_generator.py:484-498`) — 그다음 이미지법이 교점을 해석적으로 다시 푼다(`image_method.py:37-47`).
2. 그래서 광선은 **정찰병**이고 답의 단위는 경로다. 광선 수를 10,000 [^1] 발에서 4,000,000 [^2] 발까지 400 [^3]배 올려도 정반사 진폭 스프레드는 0.0 dB [^4] 다.
3. 엔진은 자기가 푸는 문제를 정확히 푼다 — 평면 반사 진폭이 이미지-소스 해석해 대비 0.9997 [^5] 이고, 자유공간 직접파가 Friis 이론과 6.3e-07 dB [^6] 안이다.
4. 설치본 2.0.1 [^7] 가 정확히 계산하는 메커니즘 7가지를 근거 줄 번호와 함께 아래 표에 그대로 싣는다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경로 탐색의 알고리즘 | 설치본 소스를 직접 읽고 `inspect.signature` 로 런타임에서 재확인했다 |
| 광선 수 무응답 | 빈 씬에서 광선 수만 바꾸며 정반사 진폭을 재는 실행 프로브 |
| 이미지-소스 대조 | 같은 형상의 해석해를 닫힌형으로 계산해 진폭 비를 잰다 |

### 재현

```bash
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_sionna_anatomy.json`, `outputs/report00_sionna_probe.json`, `outputs/report00_evidence.json` |
| 소요 | 약 1분 (GPU 0장 — JSON 읽기다) |
| 비고 | anatomy · probe 두 JSON 은 설치본 해부와 실행 프로브의 산출물이고, 각 파일의 `_meta` 가 자기 생성기 경로를 들고 있다 |

---

## 광선은 정찰병이고, 답의 단위는 경로다

답은 **두 단계**로 만들어진다. **① 경로 탐색.** 소스마다 광선을 구면에 뿌려(피보나치 격자, 난수가 아니다) 어느 면들을 어떤 순서로 맞는지 후보를 모은다. 같은 면 순서를 발견한 광선은 첫 발만 남기고 나머지를 버린다 — 면 해시로 만든 경로 지문의 카운터를 원자적으로 올리고 `samples_counter == 0` 인 광선만 저장한다(`sb_candidate_generator.py:484-498`). 그다음 이미지법이 소스를 각 면에 거울반사시켜 교점 좌표를 해석적으로 다시 푼다(`image_method.py:37-47`).

**② 필드 계산.** 그렇게 확정된 경로 **하나**를 따라가며 진폭 a 를 만든다(`field_calculator.py`).

그래서 광선은 정찰병이고, 답의 단위는 경로다. 이 구분이 [편 03 «진폭은 국소 평면파–평면경계 해 하나로 만들어진다»](03_engine-amplitude.ipynb) 의 출발점이다.

## 비유 셋과, 그 비유가 깨지는 자리

비유는 **깨지는 자리**를 함께 적어야 쓸모가 있다 (근거 `outputs/report00_po_case.json : s0_fair_boundary.analogies_and_where_they_break`).

- **Sionna 의 표면 처리는 거울 한 장이다.** 반사 세기(Fresnel)도 방향도 맞다. ⚠ 거울은 각도만 돌려준다 — 평판을 키워도 진폭이 그대로인 것이 그 뜻이고, 그 실측이 [편 05 «면적을 1600배로 키워도 경로 진폭은 7.4e-07 dB 움직인다»](05_size-sweep.ipynb) 다.

- **PO 표면적분은 조명면에 붙은 작은 안테나들의 합이다.** 점마다 위상이 더해지므로 모양이 바뀌면 보강·상쇄가 바뀐다 — σ 가 여기서 창발한다. ⚠ 그 작은 안테나의 세기를 국소 평면 반사로 정한다. 특징 폭이 파장 아래로 가면 그 가정이 깨지고, 그 무릎이 [편 22 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다»](22_po-knee.ipynb) 다.

- **SBR 은 손전등으로 비추고 빛이 닿은 자리만 세는 것이다.** 자기가림이 공짜로 처리된다. ⚠ 손전등은 모서리에서 휘는 빛과 몸통을 감아 도는 빛을 빼놓는다 — 전자가 PTD, 후자가 크리핑파이고 그 크기가 [편 23 «커널이 아직 못 하는 것은 편파 분리·PTD·재테셀레이션·다중반사 Γ(θ) 넷이고, 각각의 크기를 적었다»](23_kernel-open-items.ipynb) 다.

## Sionna 가 정확히 계산하는 것 — 먼저 이것부터

| 메커니즘 | 코드 | 무엇을 어떻게 |
|---|---|---|
| 정반사 세기 | radio_material.py:560-562, 853-892 | ITU-R P.2040 단층 슬래브 Fresnel r_te/r_tm 을 정확히 구현. 편파는 Jones 행렬로 완전히 처리. |
| 투과(굴절) | path_solver.py:153 (기본 True) | 두께 d 를 반영한 슬래브 투과계수. 단 광선은 꺾이지 않고 직진(얇은 벽 가정, path_solver.py:36-41 이 명시). |
| 가림·그림자 | Mitsuba 광선-삼각형 교차 + image_method.py:41-47 역추적 검증 | 유한 기하로 정확히 판정. 표적이 벽 뒤에 있으면 제대로 사라진다. |
| 1차 UTD 쐐기 회절 | radio_material.py:964-1144 | Kouyoumjian-Pathak + Luebbers 유한도전율. 기본값 off 일 뿐 구현은 완비. |
| 다중 반사·기하 | path_solver.py:146 max_depth=3 | 임의 순서의 반사/투과/확산 조합 경로. |
| 지연·도플러 | field_calculator.py:355, 526- | τ = 경로길이/c. 도플러는 객체당 강체 속도 1벡터 기준. |
| 확산산란(경험모델) | radio_material.py:914-962 | 거친 표면의 에너지 분산. S 로 정반사와 배분. 단 기본 S=0. |

출처 [^8]

## 광선 수는 진폭에 안 들어간다 — 다만 단서 하나

광선 수를 10,000 [^1] 발에서 4,000,000 [^2] 발까지 400 [^3]배 올려도 정반사 진폭 스프레드는 0.0 dB [^4] 다. 광선은 경로를 **찾는** 데 쓰이고 진폭을 **만드는** 데는 안 쓰인다.

⭐ 공정하게 단서를 붙인다. 이 진술은 정반사·투과·회절 경로에서 참이다. 확산산란 경로에서는 다르다 — `solid_angle` 이 4π/N 으로 초기화돼 재질까지 실려 가고, 진폭에 sqrt(fs · solid_angle) 로 곱해져 1/sqrt(N) 으로 스케일된다. 그것이 몬테카를로 추정량의 올바른 정규화다: 경로 하나는 작아지고 경로 개수가 N 에 비례해 늘어 총 전력이 수렴한다.

그 4π/N 은 **첫 상호작용에만** 살아 있고, 확산이 샘플링되는 순간 2π(반구 입체각)로 덮어써진다(근거 `outputs/report00_sionna_anatomy.json : item1_ray_shooting_and_dedup.d_ray_count_in_amplitude`).

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 확정된 경로 하나가 진폭 하나가 되는 식을 인자까지 편다 | spreading factor 와 Jones 행렬이 각각 무엇을 담당하는지가 확정된다 | [편 03 «진폭은 국소 평면파–평면경계 해 하나로 만들어…»](03_engine-amplitude.ipynb) |
| 그 식의 인자 목록을 전수로 세어 밖에 있는 양을 적는다 | 면적·곡률·치수·λ 가 목록 안인지 밖인지가 행 단위로 확정된다 | [편 04 «필드 갱신 인자 여덟 개에 면적·곡률·치수·λ…»](04_eight-factors.ipynb) |
| 같은 광선엔진 위에 면적분을 얹은 우리 커널과 맞댄다 | 가림 판정과 면적분의 분담선이 코드 경계로 확정된다 | [편 18 «가림 판정은 Sionna 광선엔진이 하고»](18_kernel-what.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 8개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report00_sionna_probe.json` | `exp_a_ray_count_sweep[0].samples_per_src` | 10000 |
| [^2] | `outputs/report00_sionna_probe.json` | `exp_a_ray_count_sweep[-1].samples_per_src` | 4000000 |
| [^3] | `outputs/report00_sionna_probe.json` | `summary.ray_count_span` | 400 |
| [^4] | `outputs/report00_sionna_probe.json` | `summary.abs_a_spread_over_ray_count_db` | 0 |
| [^5] | `outputs/report00_sionna_probe.json` | `exp_c_spreading_check.ratio_measured_over_predicted` | 0.9997 |
| [^6] | `outputs/report00_evidence.json` | `F_what_sionna_gets_right.numbers.los_agreement_db` | 6.344e-07 |
| [^7] | `outputs/report00_sionna_anatomy.json` | `item6_versions.values.sionna_rt` | 2.0.1 |
| [^8] | `outputs/report00_sionna_anatomy.json` | `item9_verdict.can_do` | (7행 표) |